### **Telco Customer Churn Predictor - Encoding and Scaling**

**Objective:** To develop a machine learning classification model that predicts customers who are likely to discontinue their telecom subscription services. The project aims to identify the key factors influencing customer churn through data cleaning, feature engineering, model training, and evaluation, enabling telecom companies to implement targeted customer retention strategies and reduce revenue loss.

**Guiding Principles for this Notebook:**  

+ Load and Inspect data
+ Split Data
+ Encode and Scale


**Import Libraries**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from src.features import (NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET)
from src.preprocessing import create_preprocessor

# Variance Inflation Factor (VIF) is a measure of multicollinearity among features in a dataset.
from statsmodels.stats.outliers_influence import variance_inflation_factor

**Load and Inspect Data**

In [2]:
data = pd.read_csv("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\telco_customer_churn_predictors.csv")
df = data.copy()

print(f"Data Shape: {df.shape}")

Data Shape: (7043, 16)


In [3]:
# replace "No internet service" with "No" for the following columns
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in service_cols:
    df[col] = df[col].replace(
        "No internet service",
        "No"
    )

In [4]:
print(f"missing values: {df.isnull().sum().sum()}")
print(f"duplicate values: {df.duplicated().sum()}")
df.head()

missing values: 0
duplicate values: 58


,tenure,Contract,OnlineSecurity,TechSupport,InternetService,PaymentMethod,OnlineBackup,DeviceProtection,MonthlyCharges,StreamingTV,StreamingMovies,PaperlessBilling,Dependents,SeniorCitizen,Partner,Churn
0,1,Month-to-month,No,No,DSL,Electronic check,Yes,No,29.85,No,No,Yes,No,No,Yes,0
1,34,One year,Yes,No,DSL,Mailed check,No,Yes,56.95,No,No,No,No,No,No,0
2,2,Month-to-month,Yes,No,DSL,Mailed check,Yes,No,53.85,No,No,Yes,No,No,No,1
3,45,One year,Yes,Yes,DSL,Bank transfer (automatic),No,Yes,42.30,No,No,No,No,No,No,0
4,2,Month-to-month,No,No,Fiber optic,Electronic check,No,No,70.70,No,No,Yes,No,No,No,1


**Train-Test Split**

In [5]:
print(f"Numerical Features: {NUMERICAL_FEATURES}")
print(f"Categorical Features: {CATEGORICAL_FEATURES}")
print(f"Target Feature: {TARGET}")

Numerical Features: ['tenure', 'MonthlyCharges']
Categorical Features: ['SeniorCitizen', 'Partner', 'Dependents', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Target Feature: Churn


In [6]:
X = df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training Set Shape: {X_train.shape}, {y_train.shape}")
print(f"Testing Set Shape: {X_test.shape}, {y_test.shape}")

Training Set Shape: (4930, 15), (4930,)
Testing Set Shape: (2113, 15), (2113,)


In [7]:
print(f"Train_duplicates: {X_train.duplicated().sum()}, Test_duplicates: {X_test.duplicated().sum()}")

Train_duplicates: 40, Test_duplicates: 8


In [8]:
# Check if any rows in the test set exist in the training set
leakage_count = X_test.isin(X_train).all(axis=1).sum()
print(f"Number of leaked rows: {leakage_count}")

Number of leaked rows: 0


**Encode and Scale**

In [9]:
# Create the preprocessor
preprocessor = create_preprocessor()

In [10]:
# Fit the preprocessor on the training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [11]:
# Convert the processed arrays back to DataFrames
X_train_processed_df = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_processed_df = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

In [12]:
X_train_processed_df.head()

,num__tenure,num__MonthlyCharges,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_Yes,cat__OnlineBackup_Yes,cat__DeviceProtection_Yes,cat__TechSupport_Yes,cat__StreamingTV_Yes,cat__StreamingMovies_Yes,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,0.881078,0.195927,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,-1.284263,0.522755,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,-0.793997,-1.509551,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-0.344587,1.053643,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
4,-1.079985,0.308740,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [13]:
# verify scaling and encoding
X_train_processed_df.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
num__tenure,4930.0,-8.359326e-17,1.000101,-1.325118,-0.957419,-0.140309,0.921933,1.616477
num__MonthlyCharges,4930.0,1.441263e-16,1.000101,-1.544390,-0.975345,0.185973,0.831333,1.785272
cat__SeniorCitizen_Yes,4930.0,1.602434e-01,0.366869,0.000000,0.000000,0.000000,0.000000,1.000000
cat__Partner_Yes,4930.0,4.851927e-01,0.499831,0.000000,0.000000,0.000000,1.000000,1.000000
cat__Dependents_Yes,4930.0,3.024341e-01,0.459359,0.000000,0.000000,0.000000,1.000000,1.000000


**Variance Inflation Factor (VIF)**

In [14]:
vif_df = pd.DataFrame()

vif_df["Feature"] = X_train_processed_df.columns

vif_df["VIF"] = [
    variance_inflation_factor(
        X_train_processed_df.values,
        i
    )
    for i in range(X_train_processed_df.shape[1])
]

vif_df = vif_df.sort_values(
    by="VIF",
    ascending=False
)

vif_df

,Feature,VIF
1,num__MonthlyCharges,11.662216
5,cat__InternetService_Fiber optic,6.769550
6,cat__InternetService_No,5.036783
14,cat__Contract_Two year,3.329875
11,cat__StreamingTV_Yes,3.040602
12,cat__StreamingMovies_Yes,3.023588
3,cat__Partner_Yes,2.774505
15,cat__PaperlessBilling_Yes,2.765255
17,cat__PaymentMethod_Electronic check,2.678789
0,num__tenure,2.472063


**Multicollinearity Assessment (VIF)**

Variance Inflation Factor (VIF) analysis was conducted after encoding and scaling to identify redundant predictors and mitigate multicollinearity issues, particularly for linear models such as Logistic Regression.

The initial analysis revealed perfect multicollinearity arising from the `"No internet service"` categories, which were consolidated into `"No"` because the absence of internet service was already captured by the `InternetService` feature. Additionally, engineered variables derived directly from existing features (`SpendCategory`, `AvgMonthlySpend`, `TenureGroup`, and `IsLongTermContract`) were removed to avoid redundancy.

The final VIF results indicated acceptable multicollinearity levels across most predictors. `MonthlyCharges` retained a relatively high VIF (11.66), reflecting its natural association with subscribed services rather than feature duplication. Given its strong predictive importance and business relevance, it will be retained for the final modeling dataset.

All remaining features exhibited acceptable VIF values (generally below 5), indicating that the final feature set is suitable for Logistic Regression and other supervised learning algorithms.

**Saving the Processed dataset**

In [15]:
# Add target back
train_df = X_train_processed_df.copy()
train_df["Churn"] = y_train.values

test_df = X_test_processed_df.copy()
test_df["Churn"] = y_test.values

print(f"Train Set Shape: {train_df.shape}, Test Set Shape: {test_df.shape}")

Train Set Shape: (4930, 20), Test Set Shape: (2113, 20)


In [16]:
display(train_df.head(),
        test_df.head()) 

,num__tenure,num__MonthlyCharges,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_Yes,cat__OnlineBackup_Yes,cat__DeviceProtection_Yes,cat__TechSupport_Yes,cat__StreamingTV_Yes,cat__StreamingMovies_Yes,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,Churn
0,0.881078,0.195927,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0
1,-1.284263,0.522755,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
2,-0.793997,-1.509551,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
3,-0.344587,1.053643,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0
4,-1.079985,0.308740,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0


,num__tenure,num__MonthlyCharges,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_Yes,cat__OnlineBackup_Yes,cat__DeviceProtection_Yes,cat__TechSupport_Yes,cat__StreamingTV_Yes,cat__StreamingMovies_Yes,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,Churn
0,-1.284263,-1.327058,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1
1,0.349957,-1.312127,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
2,0.799367,-1.507892,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0
3,-1.284263,0.383397,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
4,1.412199,-0.472660,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0


**Save Dataset as Parquet**

In [17]:
train_df.to_parquet("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\train_processed.parquet", index=False)
test_df.to_parquet("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\test_processed.parquet", index=False)

print("Processed train and test datasets have been saved to the 'data/processed' directory.")

Processed train and test datasets have been saved to the 'data/processed' directory.


**Next:** Telco Customer Churn - Handling Class Imbalance (SMOTE): only on train data

> + **notebook:** *`06_class_imbalance.ipynb`*